# NLI verification error analysis

This notebook reads only the persisted NLI prediction files. It displays generated errors without inventing examples; unavailable confidence is left blank.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
while not (root / 'results' / 'predictions').exists() and root != root.parent:
    root = root.parent
prediction_paths = sorted((root / 'results' / 'predictions').glob('NLI_*_predictions.jsonl'))
if not prediction_paths:
    raise FileNotFoundError('Run the NLI evaluation before executing this notebook.')
frames = []
for path in prediction_paths:
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]
    frame = pd.DataFrame(rows)
    frame['experiment'] = path.name.removesuffix('_predictions.jsonl')
    frames.append(frame)
errors = pd.concat(frames, ignore_index=True)
errors = errors[errors['true_label'] != errors['predicted_label']].copy()
print(f'Persisted NLI incorrect predictions: {len(errors):,}')
display(errors[['claim', 'evidence', 'true_label', 'predicted_label', 'source_dataset']].head(5))

In [ ]:
categories = {
    'explicit contradiction': r'\b(not|no|never|false|incorrect|contrary)\b',
    'unsupported / neutral': r'\b(unknown|unclear|no evidence|cannot|not enough)\b',
    'numerical claims': r'\b\d+(?:[.,]\d+)?\b|\b(percent|million|billion)\b',
    'dates / temporal claims': r'\b(19|20)\d{2}\b|\b(today|yesterday|before|after)\b',
    'named entities': r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+\b',
    'multi-hop evidence': r'\b(therefore|because|according to|which means)\b',
    'long evidence': r'.{1000,}',
    'ambiguous wording': r'\b(may|might|could|likely|allegedly|reportedly)\b',
}
text = errors['claim'].fillna('') + ' ' + errors['evidence'].fillna('')
columns = ['claim', 'evidence', 'true_label', 'predicted_label', 'confidence', 'source_dataset', 'experiment']
for category, pattern in categories.items():
    flags = text.str.contains(pattern, regex=True, na=False, case=(category == 'named entities'))
    subset = errors[flags]
    print(f'\n## {category}: {len(subset):,} actual errors')
    if subset.empty:
        print('Insufficient examples in generated NLI predictions.')
    else:
        display(subset.reindex(columns=columns).head(10))

## Interpretation guardrails

These categories are transparent text heuristics for selecting review candidates, not ground-truth error causes. `UNSUPPORTED` means insufficient support in the provided evidence; it does not by itself establish hallucination.